In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
df = spark.read.csv("/Volumes/workspace/default/day1files/C1.csv",header=True,inferSchema=True)
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023|blank|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|   Blank|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|     blank|  450|       C|
+----------+-------+--------+----------+-----+--------+



Turn blank values into Null Values


In [0]:
df = df.replace(["blank","Blank"],None)
df.show()

+----------+-------+--------+----------+-----+--------+
|CustomerID|   Name| Country|  JoinDate|Sales|Category|
+----------+-------+--------+----------+-----+--------+
|       101|  Alice|     USA|15-01-2023|  250|       A|
|       102|    Bob|   india|01-05-2023|  150|       B|
|       103|Charlie|      UK|20-02-2023| NULL|       C|
|       104|  Alice|     USA|15-01-2023|  250|       A|
|       105|    Eve|      UK|01-03-2023|  300|    NULL|
|       106|Mallory|New York|03-15-2023|  400|       B|
|       107|  Trent|   india|10-04-2023|  350|       B|
|       108|    Bob|   India|05-01-2023|  150|       B|
|       109|  Oscar|     USA|28-02-2023|  500|       A|
|       110|  Peggy|      UK|      NULL|  450|       C|
+----------+-------+--------+----------+-----+--------+



Fix Country Column (Standardization) india / India /USA -> Inconsistent



In [0]:
df = df.withColumn("Country",upper(col("Country")))
df = df.replace("NEW YORK","USA")
df.show()

+----------+-------+-------+----------+-----+--------+
|CustomerID|   Name|Country|  JoinDate|Sales|Category|
+----------+-------+-------+----------+-----+--------+
|       102|    Bob|  INDIA|01-05-2023|  150|       B|
|       110|  Peggy|     UK|01-01-2023|  450|       C|
|       109|  Oscar|    USA|28-02-2023|  500|       A|
|       106|Mallory|    USA|03-15-2023|  400|       B|
|       103|Charlie|     UK|20-02-2023|    0|       C|
|       101|  Alice|    USA|15-01-2023|  250|       A|
|       105|    Eve|     UK|01-03-2023|  300| Unknown|
|       107|  Trent|  INDIA|10-04-2023|  350|       B|
+----------+-------+-------+----------+-----+--------+



Fill Missing Values

****

In [0]:
df = df.fillna({
    "Category":"Unknown",
    "Sales":0,
    "JoinDate":"01-01-2023"
})
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
102,Bob,INDIA,01-05-2023,150,B
110,Peggy,UK,01-01-2023,450,C
109,Oscar,USA,28-02-2023,500,A
106,Mallory,USA,03-15-2023,400,B
103,Charlie,UK,20-02-2023,0,C
101,Alice,USA,15-01-2023,250,A
105,Eve,UK,01-03-2023,300,Unknown
107,Trent,INDIA,10-04-2023,350,B


Remove Negative Sales
``

In [0]:
df = df.filter(col("Sales")>=0)
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,INDIA,01-05-2023,150,B
103,Charlie,UK,20-02-2023,0,C
105,Eve,UK,01-03-2023,300,Unknown
106,Mallory,USA,03-15-2023,400,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,01-01-2023,450,C
107,Trent,INDIA,10-04-2023,350,B


Drop Redundant Customers


In [0]:
df = df.dropDuplicates(["Name"])
df.display()

CustomerID,Name,Country,JoinDate,Sales,Category
101,Alice,USA,15-01-2023,250,A
102,Bob,INDIA,01-05-2023,150,B
103,Charlie,UK,20-02-2023,0,C
105,Eve,UK,01-03-2023,300,Unknown
106,Mallory,USA,03-15-2023,400,B
109,Oscar,USA,28-02-2023,500,A
110,Peggy,UK,01-01-2023,450,C
107,Trent,INDIA,10-04-2023,350,B


### Save the Cleaned data




In [0]:
df.write.mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/workspace/default/day1files/C1.csv")